# AURORA-VISION — Model Evaluation & Benchmarking

This notebook runs the full **BLEU-4 + ROUGE-L + METEOR + BERTScore** evaluation suite
against MSR-VTT and ActivityNet Captions.

Topics covered:
1. Load MSR-VTT / ActivityNet annotation splits
2. Run batch caption generation
3. Compute all four evaluation metrics
4. Compare AURORA-VISION vs. baselines
5. Plot score distributions and radar chart

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from utils.seed import set_seed
set_seed(42)

## 1. Load MSR-VTT Annotations

In [ ]:
from data.fetchers.msrvtt_fetcher import MSRVTTFetcher

fetcher = MSRVTTFetcher(data_root="/tmp/aurora_eval")
annotations = fetcher.load_annotations(split="test")

print(f"Loaded {len(annotations)} MSR-VTT test annotations")
print("Sample:", annotations[0])

## 2. Run Caption Generation (Batch)

We run AURORA-VISION caption generation on a sample of 100 test videos.

In [ ]:
from generation.caption_generator import CaptionGenerator
from generation.prompt_builder import PromptBuilder

gen = CaptionGenerator()
builder = PromptBuilder()

predictions = []
references  = []

# Use a small sample for demonstration
SAMPLE_SIZE = min(50, len(annotations))

for ann in annotations[:SAMPLE_SIZE]:
    # Build context from available annotation fields
    caption_obj = gen.generate_frame_caption(
        clip_similarity_scores={ann.get("category", "video"): 0.8},
        ocr_text=[],
        transcript_segment=ann.get("transcript", ""),
    )
    predictions.append(caption_obj.text)
    references.append(ann["caption"])

print(f"Generated {len(predictions)} captions")
print("\nSample prediction:", predictions[0])
print("Sample reference :", references[0])

## 3. Full Evaluation (BLEU-4 + ROUGE-L + METEOR + BERTScore)

In [ ]:
from evaluation.caption_evaluator import MultiModalEvaluator

evaluator = MultiModalEvaluator()
report = evaluator.full_evaluation(predictions, references)

print("=" * 50)
print(f"  BLEU-4    : {report.scores['bleu']:.4f}")
print(f"  ROUGE-L   : {report.scores['rouge']:.4f}")
print(f"  METEOR    : {report.scores['meteor']:.4f}")
print(f"  BERTScore : {report.scores['bertscore']:.4f}")
print(f"  AGGREGATE : {report.aggregate:.4f}")
print("=" * 50)

## 4. Per-Sample Score Distribution

In [ ]:
from rouge_score import rouge_scorer as rs

scorer = rs.RougeScorer(["rougeL"], use_stemmer=True)
per_sample_rouge = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(predictions, references)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of ROUGE-L scores
axes[0].hist(per_sample_rouge, bins=20, color="#00bcd4", edgecolor="white")
axes[0].axvline(np.mean(per_sample_rouge), color="red", linestyle="--", label=f"Mean={np.mean(per_sample_rouge):.3f}")
axes[0].set_title("ROUGE-L Score Distribution")
axes[0].set_xlabel("ROUGE-L F1")
axes[0].set_ylabel("Count")
axes[0].legend()

# Bar chart of all 4 metrics
metrics = ["BLEU-4", "ROUGE-L", "METEOR", "BERTScore"]
values  = [report.scores["bleu"], report.scores["rouge"],
           report.scores["meteor"], report.scores["bertscore"]]
colors  = ["#4caf50", "#2196f3", "#ff9800", "#9c27b0"]
axes[1].bar(metrics, values, color=colors, edgecolor="white")
axes[1].set_title("AURORA-VISION Caption Evaluation")
axes[1].set_ylabel("Score (0–1)")
axes[1].set_ylim(0, 1)
for i, v in enumerate(values):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

## 5. Benchmark Against Baselines (Radar Chart)

In [ ]:
categories = ["BLEU-4", "ROUGE-L", "METEOR", "BERTScore"]

aurora_scores   = [report.scores[k] for k in ["bleu", "rouge", "meteor", "bertscore"]]
baseline_scores = [0.12, 0.22, 0.18, 0.80]   # Example SOTA baseline numbers

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=aurora_scores + [aurora_scores[0]],
    theta=categories + [categories[0]],
    fill="toself",
    name="AURORA-VISION",
    line=dict(color="#00bcd4"),
))
fig.add_trace(go.Scatterpolar(
    r=baseline_scores + [baseline_scores[0]],
    theta=categories + [categories[0]],
    fill="toself",
    name="Baseline (SOTA)",
    line=dict(color="#ff7043"),
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="AURORA-VISION vs. Baseline — Evaluation Radar",
    showlegend=True,
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(color="white"),
)
fig.show()

## 6. QA Evaluation with RetrievalEvaluator

In [ ]:
from evaluation.retrieval_evaluator import RetrievalEvaluator
import torch

evaluator_ret = RetrievalEvaluator()

# Simulate a retrieval experiment with random embeddings
N = 50
video_embeddings = torch.randn(N, 512)
text_embeddings  = torch.randn(N, 512)

metrics = evaluator_ret.compute_retrieval_metrics(video_embeddings, text_embeddings)

print("Retrieval Evaluation (V→T / T→V):")
for key, val in metrics.items():
    print(f"  {key}: {val:.4f}")